# Empirical Control-Portfolio Audit

## tl;dr

Exhaustive evaluation of all **64 control portfolios** corrects the earlier additive
recommendation. Under budget 40, completion >= 85%, and review load <= 30%, the
optimum is **Context Envelope + Tool Version Lock + Permission Scope** (cost 27),
reducing stressed incidents from **62.21% to 33.71%**. The result remains between
32.00% and 35.86% across 12 seeds.

## Context & Methods

Each portfolio is evaluated on 200 tasks x 7 non-null stressors = 1,400 runs. The
grid contains every subset of six controls. Shapley values use all coalitions, pair
interactions compare observed risk with an independent-risk expectation, and seed
sensitivity repeats the stressed evaluation for 12 consecutive seeds.

### Key Assumptions

- Governance cost and control mechanics are synthetic project parameters.
- Portfolio comparisons are empirical within the simulator, not causal claims about production.
- Default feasibility requires cost <= 40, completion >= 85%, and review load <= 30%.

## Data

### 1. Load and reconcile the full portfolio grid

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
assert (ROOT / "pyproject.toml").exists(), "Run from the repository root"
control_dir = ROOT / "data" / "control_science"
grid = pd.read_csv(control_dir / "control_portfolio_grid.csv")
by_workflow = pd.read_csv(control_dir / "control_portfolio_by_workflow.csv")
shapley = pd.read_csv(control_dir / "control_shapley.csv")
interactions = pd.read_csv(control_dir / "control_interactions.csv")
sensitivity = pd.read_csv(control_dir / "seed_sensitivity.csv")
assert len(grid) == 64 and grid["portfolio"].nunique() == 64
assert len(by_workflow) == 64 * 4
assert (grid["runs"] == 1400).all()
grid.shape, by_workflow.shape

((64, 18), (256, 16))

## Results

### 2. Solve the constrained empirical optimization

In [2]:
feasible = grid[(grid["cost"] <= 40) & (grid["task_success_rate"] >= .85) & (grid["human_review_load"] <= .30)]
optimum = feasible.sort_values(["incident_rate", "cost", "task_success_rate"], ascending=[True, True, False]).iloc[0]
optimum[["portfolio", "cost", "incident_rate", "risk_reduction", "task_success_rate", "human_review_load", "worst_workflow_incident_rate"]]

portfolio                       context_envelope+permission_scope+tool_version...
cost                                                                         27.0
incident_rate                                                            0.337143
risk_reduction                                                              0.285
task_success_rate                                                        0.854286
human_review_load                                                        0.276429
worst_workflow_incident_rate                                             0.428571
Name: 22, dtype: object

In [3]:
frontier = grid[grid["pareto_efficient"]].sort_values("cost")
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.scatter(grid.loc[~grid["feasible_default"], "cost"], grid.loc[~grid["feasible_default"], "incident_rate"], facecolors="none", edgecolors="#94A3B8", label="Infeasible")
ax.scatter(grid.loc[grid["feasible_default"], "cost"], grid.loc[grid["feasible_default"], "incident_rate"], color="#2563EB", label="Feasible")
ax.scatter(frontier["cost"], frontier["incident_rate"], facecolors="none", edgecolors="#D4A72C", s=90, label="3-objective Pareto set")
ax.scatter([optimum["cost"]], [optimum["incident_rate"]], color="#E87722", s=100, marker="*", label="Budget-40 optimum")
ax.set(xlabel="Governance cost", ylabel="Stressed incident rate", title="All 64 empirical portfolios")
ax.legend(frameon=False)
plt.tight_layout()

### 3. Reconcile marginal attribution

In [4]:
baseline_risk = float(grid.loc[grid["portfolio"] == "none", "incident_rate"].iloc[0])
full_risk = float(grid.loc[grid["control_count"] == 6, "incident_rate"].iloc[0])
assert abs(shapley["shapley_risk_reduction"].sum() - (baseline_risk - full_risk)) < 1e-10
shapley[["label", "shapley_risk_reduction", "cost", "shapley_per_cost"]].round(4)

,label,shapley_risk_reduction,cost,shapley_per_cost
0,Selective Human Review,0.1816,45.0,0.0040
1,Context Envelope,0.0896,12.0,0.0075
2,External Isolation,0.0611,14.0,0.0044
3,Permission Scope,0.0573,10.0,0.0057
4,Tool Version Lock,0.0489,5.0,0.0098
5,Rollback Hook,0.0322,18.0,0.0018


### 4. Identify complementarity and diminishing returns

In [5]:
strongest = interactions.nlargest(3, "synergy")[["label_a", "label_b", "synergy", "interpretation"]]
weakest = interactions.nsmallest(3, "synergy")[["label_a", "label_b", "synergy", "interpretation"]]
print("Strongest complementarity")
display(strongest.round(4))
print("Strongest diminishing returns")
display(weakest.round(4))

Strongest complementarity


,label_a,label_b,synergy,interpretation
0,Context Envelope,Permission Scope,0.0161,complementary
1,Context Envelope,Tool Version Lock,0.0140,complementary
2,Tool Version Lock,External Isolation,0.0115,complementary


Strongest diminishing returns


,label_a,label_b,synergy,interpretation
14,Context Envelope,Selective Human Review,-0.0114,diminishing_returns
13,Permission Scope,External Isolation,-0.0066,diminishing_returns
12,Selective Human Review,External Isolation,-0.0059,diminishing_returns


### 5. Quantify seed sensitivity

In [6]:
seed_summary = sensitivity.groupby("configuration").agg(
    mean=("incident_rate", "mean"),
    standard_deviation=("incident_rate", "std"),
    minimum=("incident_rate", "min"),
    maximum=("incident_rate", "max"),
)
seed_summary.round(4)

,mean,standard_deviation,minimum,maximum
configuration,,,,
empirical_budget_40,0.3369,0.0114,0.3200,0.3586
none,0.6054,0.0091,0.5893,0.6221
recommended_bundle,0.3369,0.0114,0.3200,0.3586


## Takeaways

1. Exact joint testing changes the recommendation: Tool Version Lock replaces External
   Isolation, lowering cost from 36 to 27 while retaining the measured 28.5-point reduction.
2. Selective Human Review has the largest average marginal Shapley value but costs 45,
   so it is infeasible under the default budget. Tool Version Lock leads per cost.
3. Context Envelope + Permission Scope is the strongest complementary pair (+0.0161),
   while Context Envelope + Selective Human Review has the strongest diminishing return.
4. The chosen bundle is stable in this simulator across 12 seeds, but the assumptions—not
   only random variation—must be challenged before production use.